# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# Probability

This notebook accompanies Chapter 1, Sections 1.1--1.3. We will:

- specify finite probability models;
- keep outcomes, singleton events, and general events distinct;
- calculate conditional probabilities and use Bayes' formula;
- check independence in a fully specified model; and
- use simulation as evidence about a model, not as the definition of probability.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(101)


## Start with a simple finite model

For a fair six-sided die, the sample space is
$\Omega=\{1,2,3,4,5,6\}$, the event collection is
$\mathcal F=2^\Omega$, and every outcome has probability $1/6$.

An outcome such as $\omega=4$ is an element of $\Omega$. The singleton
$\{4\}$ and the event $A=\{2,4,6\}$ are subsets of $\Omega$. Thus we
write $\mathbb P(\{4\})=1/6$, while
$\mathbb P(A)=3/6$.


In [ ]:
omega = set(range(1, 7))
pmf = {outcome: 1 / 6 for outcome in omega}


def event_probability(event, probability_mass):
    event = set(event)
    if not event <= set(probability_mass):
        raise ValueError("The event must be a subset of the sample space.")
    masses = np.array(list(probability_mass.values()), dtype=float)
    if np.any(masses < 0) or not np.isclose(masses.sum(), 1.0):
        raise ValueError("The probability masses must be nonnegative and sum to 1.")
    return sum(probability_mass[outcome] for outcome in event)


A = {2, 4, 6}
B = {4, 5, 6}
print("P({4}) =", event_probability({4}, pmf))
print("P(A) =", event_probability(A, pmf))
print("P(A union B) =", event_probability(A | B, pmf))
print("P(A) + P(B) - P(A intersection B) =",
      event_probability(A, pmf) + event_probability(B, pmf)
      - event_probability(A & B, pmf))


A probability measure satisfies $\mathbb P(\Omega)=1$, is nonnegative,
and is **countably additive** on pairwise disjoint events. In a finite model,
the probability of an event is the sum of the masses of its outcomes. Finite
additivity is a consequence of countable additivity; it is not a replacement
for the standard definition.


## Updating probabilities with new information

Conditional probability is defined only when the conditioning event has
positive probability:

$$
\mathbb P(A\mid B)=\frac{\mathbb P(A\cap B)}{\mathbb P(B)},
\qquad \mathbb P(B)>0.
$$

Consider a diagnostic test. Let $D$ be the event that a person has a
condition and $+$ the event of a positive test. Specify

$$
\mathbb P(D)=0.01,\qquad
\mathbb P(+\mid D)=0.95,\qquad
\mathbb P(+\mid D^c)=0.10.
$$

The two events $D,D^c$ form a partition, so the law of total probability
and Bayes' formula apply.


In [ ]:
prevalence = 0.01
sensitivity = 0.95
false_positive_rate = 0.10

p_positive = (sensitivity * prevalence
              + false_positive_rate * (1 - prevalence))
p_condition_given_positive = sensitivity * prevalence / p_positive

print(f"P(positive) = {p_positive:.4f}")
print(f"P(condition | positive) = {p_condition_given_positive:.4f}")


## When are events independent?

For two independent tosses of a fair coin, use
$\Omega=\{HH,HT,TH,TT\}$, with mass $1/4$ at every outcome. Let $A$
be “the first toss is Heads” and $B$ be “the second toss is Heads.” Then
$A$ and $B$ are independent because
$\mathbb P(A\cap B)=\mathbb P(A)\mathbb P(B)$. Repeating an experiment
does not by itself establish independence; independence is an assumption in
this model.


In [ ]:
coin_pmf = {"HH": 0.25, "HT": 0.25, "TH": 0.25, "TT": 0.25}
first_heads = {"HH", "HT"}
second_heads = {"HH", "TH"}

p_a = event_probability(first_heads, coin_pmf)
p_b = event_probability(second_heads, coin_pmf)
p_intersection = event_probability(first_heads & second_heads, coin_pmf)
print("P(A intersection B) =", p_intersection)
print("P(A) P(B) =", p_a * p_b)
print("Independent?", np.isclose(p_intersection, p_a * p_b))


## Checking a model with relative frequency

Under the stated model of independent fair die rolls, the law of large
numbers later explains why the relative frequency of sixes approaches
$1/6$ with high probability. The plot below illustrates one simulated
path. It does not define $\mathbb P$, and one path is not a proof.


In [ ]:
n_rolls = 5000
rolls = rng.integers(1, 7, size=n_rolls)
running_frequency = np.cumsum(rolls == 6) / np.arange(1, n_rolls + 1)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(np.arange(1, n_rolls + 1), running_frequency, label="simulated frequency")
ax.axhline(1 / 6, color="black", linestyle="--", label="model probability 1/6")
ax.set(xlabel="number of rolls", ylabel="fraction of sixes", ylim=(0, 0.35))
ax.legend()
plt.show()


## Try it yourself

1. For the fair-die model, let $C=\{1,2,3\}$ and $D=\{2,3,4\}$.
   Compute $\mathbb P(C\mid D)$, checking first that the denominator is
   positive.
2. In the diagnostic-test model, recompute
   $\mathbb P(D\mid +)$ if the prevalence is $0.10$. Explain why the
   sensitivity and false-positive rate alone do not determine the answer.
3. Find two events in the two-toss model that are not independent and verify
   the failed product identity.
